# HASEM v2 — leakage-safe healthy-reference anomaly detection

Pan-cancer transcriptomic anomaly detection with PCA, Hotelling's T² and Q-residual scores.

**Design**
- Every fitted step (gene selection, imputation, scaling, PCA, control limits) uses only the partition it is allowed to see.
- Healthy samples are split 70/15/15 by patient/donor group: **train** (gene selection, imputation, scaling, PCA), **calibration** (control limits) and **holdout** (final evaluation only).
- Tumour samples are never used to fit or calibrate anything.
- Headline numbers are computed on the evaluation pool only (healthy holdout + tumours).
- Parameters match v1 (5,000 genes, 99th percentile), so the leakage fix is the only change.

**Cells**

| Cell | Content |
|---|---|
| 0 | Environment, constants, design switches, helpers |
| 1 | Metadata, labels, patient-aware partitions, train-only gene selection and preprocessing |
| 1B | Optional batch handling and PCA |
| 2 | Provenance and batch structure |
| 3 | T² and Q scoring, control limits, headline results, ablation, false alarms |
| 4 / 4B | Per-cancer detection rates; cell-line sensitivity |
| 5 / 5C | Leave-one-cancer-out validation; true LOCO |
| 6 / 7 | Supervised and multiclass classification with patient-group splits |
| 8 | Gene contribution mapping and pathway enrichment |
| 9 | Final summary |

Outputs are written to `OUTDIR` (default `results_hasem_v2`).

## Cell 0 — Environment and settings

Dependencies, imports, reproducibility constants, design switches, paths, helper functions and a configuration snapshot.

In [1]:
# Install dependencies (Kaggle/Colab); skip if already installed.
get_ipython().getoutput("pip install -q gprofiler-official mygene scikit-learn xgboost shap")

[]

### 0a — Imports

In [2]:
import os, sys, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f as f_dist, binom

from gprofiler import GProfiler
import mygene

from sklearn.impute          import SimpleImputer
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.decomposition   import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, IsolationForest
from sklearn.calibration     import calibration_curve
from sklearn.metrics         import (
    roc_auc_score, average_precision_score, accuracy_score,
    f1_score, recall_score, confusion_matrix, brier_score_loss,
    RocCurveDisplay, PrecisionRecallDisplay, classification_report
)

try:
    import xgboost as xgb
    HAVE_XGB = True
except ImportError:
    HAVE_XGB = False
try:
    import shap
    HAVE_SHAP = True
except ImportError:
    HAVE_SHAP = False

### 0b — Reproducibility, constants and design switches

`COMBAT_MODE` (`none` | `source_shift` | `global_original`) and `EVAL_PRIMARY_ONLY` are set here.

In [3]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

CHUNK_SIZE          = 5000
TOP_K               = int(os.environ.get("HASEM_TOP_K", 5000))   # MAD-selected genes
VAR_EXPLAINED       = 0.95      # PCA cumulative variance threshold
LOW_EXPR_THRESHOLD  = 1.0       # minimum median expression filter
ANOMALY_ALPHA       = 0.01      # significance level for the theoretical limits
MIN_CANCER_SAMPLES  = 5         # minimum samples per cancer type

SPLIT_FRACTIONS   = (0.70, 0.15, 0.15)  # healthy: train / calibration / holdout
THRESH_PCT        = 99.0        # percentile of healthy-CALIBRATION scores used as control limit
COMBAT_MODE       = "none"  # batch handling: "none" (default) | "source_shift" | "global_original" (v1, not leakage-safe)
EVAL_PRIMARY_ONLY = False  # True = evaluate on primary tumours only (drops cell lines, metastatic, recurrent)
LOCO_B_REFIT_EACH = False  # LOCO-B reference is identical for every cancer; False computes it once

### 0c — Warnings, paths and output folders

In [4]:
warnings.filterwarnings("ignore")
try:
    pd.set_option("future.no_silent_downcasting", True)
except Exception:
    pass

EXPR_PATH  = os.environ.get("HASEM_EXPR_PATH",
    "/kaggle/input/datasets/kaythie/cancer-dataset/TcgaTargetGtex_gene_expected_count/TcgaTargetGtex_gene_expected_count")
PHENO_PATH = os.environ.get("HASEM_PHENO_PATH",
    "/kaggle/input/datasets/kaythie/cancer-dataset/TcgaTargetGTEX_phenotype.txt/TcgaTargetGTEX_phenotype.txt")

OUTDIR = os.environ.get("HASEM_OUTDIR", "results_hasem_v2")
for sub in ["preprocessing","batch_effects","unsupervised","loco",
            "calibration","supervised","per_cancer","pathway_enrichment",
            "contributions","models","logs"]:
    os.makedirs(os.path.join(OUTDIR, sub), exist_ok=True)

HEALTHY_TAGS = ["GTEX", "Normal Tissue", "Solid Tissue Normal"]

### 0d — Helper functions

In [5]:
def log(msg):
    print(f"[HASEM] {msg}")

def wilson_ci(k, n, z=1.96):
    """Wilson score 95% confidence interval for a proportion, in percent."""
    if n == 0:
        return np.nan, np.nan
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    margin = (z * np.sqrt(p*(1-p)/n + z**2/(4*n**2))) / denom
    return round(max(0.0, (centre - margin)*100), 2), round(min(100.0, (centre + margin)*100), 2)

def compute_T2(Xpca, eigenvals):
    safe = np.where(eigenvals <= 0, 1e-12, eigenvals)
    return np.sum((Xpca**2) / safe, axis=1)

### 0e — Save configuration snapshot

In [6]:
pd.Series({
    "RANDOM_SEED": RANDOM_SEED, "TOP_K": TOP_K, "VAR_EXPLAINED": VAR_EXPLAINED,
    "LOW_EXPR_THRESHOLD": LOW_EXPR_THRESHOLD, "ANOMALY_ALPHA": ANOMALY_ALPHA,
    "SPLIT_FRACTIONS": str(SPLIT_FRACTIONS), "THRESH_PCT": THRESH_PCT,
    "COMBAT_MODE": COMBAT_MODE, "EVAL_PRIMARY_ONLY": EVAL_PRIMARY_ONLY,
    "LOCO_B_REFIT_EACH": LOCO_B_REFIT_EACH,
    "HAVE_XGB": HAVE_XGB, "HAVE_SHAP": HAVE_SHAP
}).to_csv(os.path.join(OUTDIR, "config_snapshot.csv"))

log("Environment ready (v2, leakage-safe).")

[HASEM] Environment ready (v2, leakage-safe).


## Cell 1 — Metadata, labels, patient-aware partitions and preprocessing

**1a. Load and align metadata.** Gene selection and preprocessing use healthy training samples only.

In [7]:
log("Step 1: metadata, patient-aware split, feature selection...")

meta = pd.read_csv(PHENO_PATH, sep="\t", encoding="latin1", low_memory=False)
meta.rename(
    columns={
        "sample_type": "_sample_type",
        "primary_site": "_primary_site",
        "Sample": "sample"
    },
    inplace=True
)
meta = meta.set_index("sample", drop=False)

expr_cols = pd.read_csv(EXPR_PATH, sep="\t", index_col=0, nrows=0).columns
meta = meta.loc[meta.index.isin(expr_cols)].copy()

[HASEM] Step 1: metadata, patient-aware split, feature selection...


### 1b — Conservative sample labels

In [8]:
sample_type = meta["_sample_type"].fillna("").astype(str).str.strip()

meta["is_healthy"] = sample_type.isin(
    ["Normal Tissue", "Solid Tissue Normal"]
)
meta["is_primary_tumour"] = sample_type.eq("Primary Tumor")
meta["eligible_primary_analysis"] = (
    meta["is_healthy"] | meta["is_primary_tumour"]
)

if "_study" in meta.columns:
    study = meta["_study"].fillna("").astype(str).str.upper()
    meta["source"] = np.select(
        [
            study.str.contains("GTEX"),
            study.str.contains("TARGET"),
            study.str.contains("TCGA")
        ],
        ["GTEx", "TARGET", "TCGA"],
        default="Unknown"
    )
else:
    meta["source"] = "Unknown"

meta = meta.loc[meta["eligible_primary_analysis"]].copy()
meta["is_tumor"] = meta["is_primary_tumour"].astype(int)

# Patient IDs: group all samples from the same participant.
sample_ids = pd.Series(meta.index.astype(str), index=meta.index)

tcga_patient = sample_ids.str.extract(
    r"^(TCGA-[A-Z0-9]+-[A-Z0-9]+)", expand=False
)
gtex_donor = sample_ids.str.extract(
    r"^(GTEX-[A-Z0-9]+)", expand=False
)

meta["patient_group"] = sample_ids.copy()

tcga_mask = meta["source"].eq("TCGA")
gtex_mask = meta["source"].eq("GTEx")

meta.loc[tcga_mask, "patient_group"] = (
    tcga_patient.loc[tcga_mask]
    .fillna(sample_ids.loc[tcga_mask])
)

meta.loc[gtex_mask, "patient_group"] = (
    gtex_donor.loc[gtex_mask]
    .fillna(sample_ids.loc[gtex_mask])
)

log(f"Eligible healthy tissues: {int(meta['is_healthy'].sum()):,}")
log(f"Eligible primary tumours: {int(meta['is_primary_tumour'].sum()):,}")
log(f"Source counts: {meta['source'].value_counts().to_dict()}")

[HASEM] Eligible healthy tissues: 8,149
[HASEM] Eligible primary tumours: 9,181
[HASEM] Source counts: {'TCGA': 9907, 'GTEx': 7412, 'TARGET': 11}


### 1b (cont.) — Healthy patient-group split and tumour exclusion, first pass

Splits healthy patient/donor groups and excludes TCGA tumours whose patients are in the healthy training or calibration groups.

In [9]:
# Split healthy patient/donor groups, not individual samples.
healthy_meta = meta.loc[meta["is_healthy"]].copy()

patient_table = (
    healthy_meta[["patient_group", "source", "_primary_site"]]
    .drop_duplicates("patient_group")
    .set_index("patient_group")
)

strata = (
    patient_table["source"].astype(str)
    + "|"
    + patient_table["_primary_site"].fillna("Unknown_Tissue").astype(str)
)

counts = strata.value_counts()
strata = strata.where(strata.map(counts) >= 3, "RARE")

groups = patient_table.index.to_numpy()
strata_values = strata.to_numpy()

f_tr, f_ca, f_ho = SPLIT_FRACTIONS

try:
    train_groups, rest_groups, _, rest_strata = train_test_split(
        groups,
        strata_values,
        test_size=f_ca + f_ho,
        random_state=RANDOM_SEED,
        stratify=strata_values
    )

    calib_groups, holdout_groups = train_test_split(
        rest_groups,
        test_size=f_ho / (f_ca + f_ho),
        random_state=RANDOM_SEED,
        stratify=rest_strata
    )
except ValueError:
    log("WARNING: stratified patient split unavailable; using random group split.")
    train_groups, rest_groups = train_test_split(
        groups,
        test_size=f_ca + f_ho,
        random_state=RANDOM_SEED
    )
    calib_groups, holdout_groups = train_test_split(
        rest_groups,
        test_size=f_ho / (f_ca + f_ho),
        random_state=RANDOM_SEED
    )

healthy_train_groups = set(train_groups)
healthy_calib_groups = set(calib_groups)
healthy_holdout_groups = set(holdout_groups)

assert not (healthy_train_groups & healthy_calib_groups)
assert not (healthy_train_groups & healthy_holdout_groups)
assert not (healthy_calib_groups & healthy_holdout_groups)

meta["partition"] = "excluded"
meta.loc[meta["is_primary_tumour"], "partition"] = "tumour"

meta.loc[
    meta["is_healthy"] & meta["patient_group"].isin(healthy_train_groups),
    "partition"
] = "healthy_train"

meta.loc[
    meta["is_healthy"] & meta["patient_group"].isin(healthy_calib_groups),
    "partition"
] = "healthy_calib"

meta.loc[
    meta["is_healthy"] & meta["patient_group"].isin(healthy_holdout_groups),
    "partition"
] = "healthy_holdout"

# Exclude TCGA tumours whose patients contributed healthy train/calibration samples (holdout donors' tumours may remain).
protected_groups = healthy_train_groups | healthy_calib_groups

tcga_tumour = (
    meta["is_primary_tumour"] & meta["source"].eq("TCGA")
)

exclude_tcga_tumour = (
    tcga_tumour & meta["patient_group"].isin(protected_groups)
)

meta.loc[
    exclude_tcga_tumour, "partition"
] = "tumour_patient_overlap_excluded"

meta.loc[
    meta["is_primary_tumour"] & ~meta["source"].eq("TCGA"),
    "partition"
] = "tumour"

[HASEM] WARNING: stratified patient split unavailable; using random group split.


### 1c — Healthy patient-group split (repeated)

Repeats the split with the same settings and **overwrites `meta['partition']`**. The partitions used from here on come from this cell and from 1d.

In [10]:
healthy_meta = meta.loc[meta["is_healthy"]].copy()

patient_table = healthy_meta[
    ["patient_group", "source", "_primary_site"]
].drop_duplicates("patient_group").set_index("patient_group")

strata = (
    patient_table["source"].astype(str) + "|" +
    patient_table["_primary_site"].fillna("Unknown_Tissue").astype(str)
)
counts = strata.value_counts()
strata = strata.where(strata.map(counts) >= 3, "RARE")

groups = patient_table.index.to_numpy()
strata_values = strata.to_numpy()

f_tr, f_ca, f_ho = SPLIT_FRACTIONS

# Fall back to a random group split if stratification is not possible.
try:
    train_groups, rest_groups, _, rest_strata = train_test_split(
        groups,
        strata_values,
        test_size=(f_ca + f_ho),
        random_state=RANDOM_SEED,
        stratify=strata_values
    )
    calib_groups, holdout_groups = train_test_split(
        rest_groups,
        test_size=f_ho / (f_ca + f_ho),
        random_state=RANDOM_SEED,
        stratify=rest_strata
    )
except ValueError:
    log("WARNING: stratified patient split unavailable; using random group split.")
    train_groups, rest_groups = train_test_split(
        groups,
        test_size=(f_ca + f_ho),
        random_state=RANDOM_SEED
    )
    calib_groups, holdout_groups = train_test_split(
        rest_groups,
        test_size=f_ho / (f_ca + f_ho),
        random_state=RANDOM_SEED
    )

healthy_train = healthy_meta.index[
    healthy_meta["patient_group"].isin(train_groups)
].to_numpy()
healthy_calib = healthy_meta.index[
    healthy_meta["patient_group"].isin(calib_groups)
].to_numpy()
healthy_holdout = healthy_meta.index[
    healthy_meta["patient_group"].isin(holdout_groups)
].to_numpy()

meta["partition"] = "excluded"
meta.loc[meta["is_primary_tumour"], "partition"] = "tumour"
meta.loc[healthy_train, "partition"] = "healthy_train"
meta.loc[healthy_calib, "partition"] = "healthy_calib"
meta.loc[healthy_holdout, "partition"] = "healthy_holdout"

[HASEM] WARNING: stratified patient split unavailable; using random group split.


### 1d — Unseen-patient tumour evaluation set

Excludes TCGA tumours from patients present in any healthy partition, including the holdout.

In [11]:
# Exclude TCGA tumours from patients present in ANY healthy partition, including the holdout.
protected_groups = (
    set(train_groups)
    | set(calib_groups)
    | set(holdout_groups)
)

tcga_tumour = (
    meta["is_primary_tumour"] & (meta["source"] == "TCGA")
)
exclude_tcga_tumour = (
    tcga_tumour & meta["patient_group"].isin(protected_groups)
)
meta.loc[exclude_tcga_tumour, "partition"] = "tumour_patient_overlap_excluded"

meta.loc[
    meta["is_primary_tumour"] & (meta["source"] != "TCGA"),
    "partition"
] = "tumour"

meta[["partition", "source", "patient_group"]].to_csv(
    os.path.join(OUTDIR, "models", "split_manifest.csv")
)

healthy_train = meta.index[meta["partition"] == "healthy_train"].to_numpy()
healthy_calib = meta.index[meta["partition"] == "healthy_calib"].to_numpy()
healthy_holdout = meta.index[meta["partition"] == "healthy_holdout"].to_numpy()
train_set = set(healthy_train)

### 1e — Variable-gene selection (healthy train only)

In [12]:
log("Computing MAD on healthy training samples only...")
mad_scores = []

with pd.read_csv(
    EXPR_PATH, sep="\t", index_col=0, chunksize=CHUNK_SIZE,
    engine="c", dtype_backend="numpy_nullable"
) as reader:
    for chunk in reader:
        train_cols = [c for c in chunk.columns if c in train_set]
        if not train_cols:
            continue

        ch = chunk[train_cols]
        ch = ch.loc[ch.median(axis=1) >= LOW_EXPR_THRESHOLD]
        if ch.empty:
            continue

        med = ch.median(axis=1)
        mad = (ch.sub(med, axis=0)).abs().median(axis=1)
        mad_scores.extend(zip(ch.index, mad))

mad_df = pd.DataFrame(mad_scores, columns=["Gene", "MAD"])
mad_df = mad_df.drop_duplicates("Gene")
mad_df["MAD_z"] = (
    (mad_df["MAD"] - mad_df["MAD"].mean()) /
    mad_df["MAD"].std()
)
top_genes = mad_df.nlargest(TOP_K, "MAD_z")["Gene"].tolist()

if not top_genes:
    raise ValueError("No genes selected. Check the expression matrix and filter.")

log(f"Selected {len(top_genes):,} genes using training samples only.")

[HASEM] Computing MAD on healthy training samples only...
[HASEM] Selected 5,000 genes using training samples only.


### 1f — Load selected genes and align samples

In [13]:
chunks = []
with pd.read_csv(
    EXPR_PATH, sep="\t", index_col=0, chunksize=CHUNK_SIZE,
    engine="c", dtype_backend="numpy_nullable"
) as reader:
    for chunk in reader:
        overlap = chunk.index.intersection(top_genes)
        if len(overlap):
            chunks.append(chunk.loc[overlap])

if not chunks:
    raise ValueError("Selected genes were not found in the expression file.")

expr_top = pd.concat(chunks).T
expr_top.index.name = "sample"

common = expr_top.index.intersection(meta.index)
expr_top = expr_top.loc[common]
meta = meta.loc[common]

log(f"Aligned matrix: {expr_top.shape[0]:,} samples x {expr_top.shape[1]:,} genes")

[HASEM] Aligned matrix: 17,330 samples x 5,000 genes


### 1g — Aligned masks

In [14]:
y_binary = meta["is_tumor"]
y_detailed = meta["_primary_site"].copy()
y_detailed.loc[meta["is_healthy"]] = "HEALTHY"

part = meta["partition"].to_numpy()
healthy_train_mask = part == "healthy_train"
healthy_calib_mask = part == "healthy_calib"
healthy_holdout_mask = part == "healthy_holdout"

eval_tumour_mask = (
    meta["partition"].eq("tumour").to_numpy()
    & meta["is_primary_tumour"].to_numpy()
)
eval_mask = healthy_holdout_mask | eval_tumour_mask

log(f"Training healthy: {healthy_train_mask.sum():,}")
log(f"Calibration healthy: {healthy_calib_mask.sum():,}")
log(f"Holdout healthy: {healthy_holdout_mask.sum():,}")
log(f"Eligible tumour evaluation: {eval_tumour_mask.sum():,}")
log(f"Excluded overlapping TCGA tumours: {exclude_tcga_tumour.sum():,}")

[HASEM] Training healthy: 5,688
[HASEM] Calibration healthy: 1,188
[HASEM] Holdout healthy: 1,273
[HASEM] Eligible tumour evaluation: 8,502
[HASEM] Excluded overlapping TCGA tumours: 679


### 1h — Preprocessing fitted on healthy train only

In [15]:
expr_numeric = expr_top.apply(pd.to_numeric, errors="coerce").astype("float64")

imputer = SimpleImputer(strategy="median", keep_empty_features=True)
imputer.fit(expr_numeric.loc[healthy_train].values)
X_imputed = pd.DataFrame(
    imputer.transform(expr_numeric.values),
    index=expr_numeric.index,
    columns=expr_numeric.columns
)

X_log = np.log2(X_imputed.clip(lower=1e-6) + 1)

scaler = StandardScaler()
scaler.fit(X_log.loc[healthy_train].values)

X_scaled = pd.DataFrame(
    scaler.transform(X_log.values),
    index=X_log.index,
    columns=X_log.columns
)
X_scaled.replace([np.inf, -np.inf], np.nan, inplace=True)
X_scaled = X_scaled.fillna(0.0)
X_scaled_original = X_scaled.copy()

### 1i — Integrity checks

Patient-overlap and donor-partition checks.

In [16]:
assert expr_top.index.equals(meta.index), "Expression/metadata index mismatch."
assert not (set(healthy_train) & set(healthy_calib))
assert not (set(healthy_train) & set(healthy_holdout))
assert not (set(healthy_calib) & set(healthy_holdout))

# No TCGA patient from healthy train/calibration may appear in the tumour evaluation set.
train_calib_groups = set(
    meta.loc[meta["partition"].isin(["healthy_train", "healthy_calib"]),
             "patient_group"]
)
eligible_tcga_eval_groups = set(
    meta.loc[
        eval_tumour_mask & (meta["source"].to_numpy() == "TCGA"),
        "patient_group"
    ]
)
assert not (train_calib_groups & eligible_tcga_eval_groups), (
    "Patient overlap remains between healthy train/calibration and TCGA tumour evaluation."
)

log("Step 1 complete. Review counts before running later cells.")

meta["patient_id"] = meta.index.astype(str).str.extract(
    r"^(TCGA-[A-Z0-9]+-[A-Z0-9]+)", expand=False
)

eval_patients = set(
    meta.loc[eval_tumour_mask, "patient_id"].dropna()
)

healthy_train_patients = set(
    meta.loc[meta["partition"].eq("healthy_train"), "patient_id"].dropna()
)
healthy_calib_patients = set(
    meta.loc[meta["partition"].eq("healthy_calib"), "patient_id"].dropna()
)
healthy_holdout_patients = set(
    meta.loc[meta["partition"].eq("healthy_holdout"), "patient_id"].dropna()
)

print("Tumour/healthy-train overlap:",
      len(eval_patients & healthy_train_patients))
print("Tumour/healthy-calibration overlap:",
      len(eval_patients & healthy_calib_patients))
print("Tumour/healthy-holdout overlap:",
      len(eval_patients & healthy_holdout_patients))

print("Final expression samples:", len(expr_top))
print("Metadata samples:", len(meta))
print("Sample indexes identical:", expr_top.index.equals(meta.index))
print("\nSample types by source:")
print(pd.crosstab(meta["source"], meta["_sample_type"]))

print("\nPartition counts:")
print(meta["partition"].value_counts(dropna=False))

print("\nTumour labels:")
print(meta.loc[meta["is_primary_tumour"], "is_tumor"].value_counts())

# TCGA barcode patient ID = first 3 barcode fields

check = meta.copy()
check["sample_id"] = check.index.astype(str)
check["patient_id"] = check["sample_id"].str.extract(
    r"^(TCGA-[A-Z0-9]+-[A-Z0-9]+)", expand=False
)

tcga = check[check["source"] == "TCGA"].copy()

tumour_patients = set(
    tcga.loc[tcga["is_primary_tumour"], "patient_id"].dropna()
)
healthy_patients = set(
    tcga.loc[tcca["is_healthy"], "patient_id"].dropna()
) if False else set(
    tcga.loc[tcga["is_healthy"], "patient_id"].dropna()
)

print("Unique TCGA tumour patients:", len(tumour_patients))
print("Unique TCGA healthy patients:", len(healthy_patients))
print("Patients appearing in both groups:",
      len(tumour_patients & healthy_patients))

import re
import pandas as pd

check = meta.copy()
check["sample_id"] = check.index.astype(str)

check["expected_donor"] = check["sample_id"]

tcga_mask = check["source"].eq("TCGA")
gtex_mask = check["source"].eq("GTEx")

check.loc[tcga_mask, "expected_donor"] = (
    check.loc[tcga_mask, "sample_id"]
    .str.extract(r"^(TCGA-[A-Z0-9]+-[A-Z0-9]+)", expand=False)
    .fillna(check.loc[tcga_mask, "sample_id"])
)

check.loc[gtex_mask, "expected_donor"] = (
    check.loc[gtex_mask, "sample_id"]
    .str.extract(r"^(GTEX-[A-Z0-9]+)", expand=False)
    .fillna(check.loc[gtex_mask, "sample_id"])
)

gtex = check.loc[gtex_mask]
print("GTEx samples:", len(gtex))
print("Unique GTEx donors:", gtex["expected_donor"].nunique())
print("GTEx donors with multiple samples:",
      int((gtex["expected_donor"].value_counts() > 1).sum()))

print("GTEx samples whose patient_group differs from donor ID:",
      int((gtex["patient_group"].astype(str) !=
           gtex["expected_donor"].astype(str)).sum()))

healthy = check[
    check["partition"].isin(
        ["healthy_train", "healthy_calib", "healthy_holdout"]
    )
].copy()

donor_partition_counts = (
    healthy.groupby("expected_donor")["partition"].nunique()
)
cross_partition_donors = donor_partition_counts[
    donor_partition_counts > 1
]

print("Healthy donors appearing in multiple partitions:",
      len(cross_partition_donors))
print(cross_partition_donors.head(10))

[HASEM] Step 1 complete. Review counts before running later cells.
Tumour/healthy-train overlap: 0
Tumour/healthy-calibration overlap: 0
Tumour/healthy-holdout overlap: 0
Final expression samples: 17330
Metadata samples: 17330
Sample indexes identical: True

Sample types by source:
_sample_type  Normal Tissue  Primary Tumor  Solid Tissue Normal
source                                                         
GTEx                   7412              0                    0
TARGET                    0              0                   11
TCGA                      0           9181                  726

Partition counts:
partition
tumour                             8502
healthy_train                      5688
healthy_holdout                    1273
healthy_calib                      1188
tumour_patient_overlap_excluded     679
Name: count, dtype: int64

Tumour labels:
is_tumor
1    9181
Name: count, dtype: int64
Unique TCGA tumour patients: 9181
Unique TCGA healthy patients: 726
Patients appe

## Cell 1B — Optional batch handling and PCA

`COMBAT_MODE='none'` (default) applies no correction; `source_shift` subtracts a per-gene TCGA/TARGET-normal vs GTEx offset estimated on healthy train only; `global_original` reproduces v1 and is not leakage-safe.

In [17]:
combat_output = None
if COMBAT_MODE == "global_original":
    # v1 behaviour, NOT leakage-safe: ComBat fitted on all samples with the tumour/healthy label as batch.
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "neuroCombat"], check=False)
    from neuroCombat import neuroCombat
    log("Step 1B-i: ComBat (LEGACY global mode — not leakage-safe)...")
    batch_labels = (meta["cohort"] == "TCGA/TARGET").astype(int).values
    covars = pd.DataFrame({"batch": batch_labels, "is_tumor": y_binary.values})
    try:
        combat_output = neuroCombat(dat=X_scaled.values.T, covars=covars, batch_col="batch",
                                    categorical_cols=["is_tumor"], mean_only=True)
        X_scaled_original = X_scaled.copy()
        X_scaled = pd.DataFrame(combat_output["data"].T, index=X_scaled.index, columns=X_scaled.columns)
        X_scaled.replace([np.inf, -np.inf], np.nan, inplace=True)
        X_scaled = X_scaled.fillna(0.0)
    except Exception as e:
        log(f"  Legacy ComBat could not run ({type(e).__name__}: {e}). The batch label equals the protected "
            f"tumour/healthy covariate, so the design is degenerate. Continuing WITHOUT correction.")
elif COMBAT_MODE == "source_shift":
    log("Step 1B-i: source-shift correction (per-gene mean shift TCGA/TARGET-normal vs GTEx, fitted on healthy-train only)...")
    src_tcga = (meta["source"].values == "TCGA/TARGET")
    g_rows = healthy_train_mask & ~src_tcga
    t_rows = healthy_train_mask & src_tcga
    if g_rows.sum() >= 20 and t_rows.sum() >= 20:
        offset = X_scaled.values[t_rows].mean(axis=0) - X_scaled.values[g_rows].mean(axis=0)
        Xs = X_scaled.values.copy()
        Xs[src_tcga] -= offset                      # applied to every TCGA/TARGET-source sample, incl. tumours
        X_scaled_original = X_scaled.copy()
        X_scaled = pd.DataFrame(Xs, index=X_scaled.index, columns=X_scaled.columns)
        log(f"  Shift estimated from {int(g_rows.sum())} GTEx and {int(t_rows.sum())} TCGA/TARGET-normal training samples.")
    else:
        log("  Not enough healthy-train samples in one source — correction skipped.")
else:
    log("Step 1B-i: ComBat NOT applied (COMBAT_MODE='none'). Batch structure is measured directly in Cell 2.")

[HASEM] Step 1B-i: ComBat NOT applied (COMBAT_MODE='none'). Batch structure is measured directly in Cell 2.


### 1B-ii — PCA fitted on healthy train only

In [18]:
log("Step 1B-ii: fitting PCA on healthy-train samples only...")
pca_unsup = PCA(n_components=VAR_EXPLAINED, svd_solver="full", random_state=RANDOM_SEED)
pca_unsup.fit(X_scaled.values[healthy_train_mask])
X_pca = pca_unsup.transform(X_scaled.values)
n_comp = pca_unsup.n_components_
log(f"  PCA components retained: {n_comp} (>= {VAR_EXPLAINED*100:.0f}% variance)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.cumsum(pca_unsup.explained_variance_ratio_), color="#1F4E79", lw=1.5)
ax.axhline(VAR_EXPLAINED, color="red", ls="--", lw=1, label=f"{VAR_EXPLAINED*100:.0f}% threshold")
ax.set_xlabel("Principal Components"); ax.set_ylabel("Cumulative Variance Explained")
ax.set_title("PCA scree — healthy-train samples only")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "preprocessing", "pca_scree.png"), dpi=150)
plt.close()

meta.to_csv(os.path.join(OUTDIR, "models", "meta_aligned.csv"))
joblib.dump({"imputer": imputer, "scaler": scaler, "pca_unsup": pca_unsup,
             "top_genes": top_genes, "combat_output": combat_output},
            os.path.join(OUTDIR, "models", "preproc_artefacts.joblib"))
log("Step 1B complete.")

[HASEM] Step 1B-ii: fitting PCA on healthy-train samples only...
[HASEM]   PCA components retained: 1330 (>= 95% variance)
[HASEM] Step 1B complete.


## Cell 2 — Provenance and batch structure

**2A.** 2-D PCA coloured by true source and disease status. **2B.** Can a classifier separate GTEx from TCGA/TARGET normal tissue? (Healthy samples only; trained on healthy train, scored on healthy holdout.)

In [19]:
log("Step 2: provenance / batch-structure analysis...")
batch_dir = os.path.join(OUTDIR, "batch_effects")

pca2 = PCA(n_components=2, random_state=RANDOM_SEED).fit(X_scaled.values[healthy_train_mask])
X_pca_2d = pca2.transform(X_scaled.values)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
panels = [("Coloured by source", meta["source"].values),
          ("Coloured by tumour / healthy", np.where(y_binary.values == 0, "Healthy", "Tumour"))]
palette = {"GTEx": "#2196F3", "TCGA/TARGET": "#F44336", "Healthy": "#4CAF50", "Tumour": "#F44336"}
for ax, (title, labels) in zip(axes, panels):
    for lab in np.unique(labels):
        m_ = labels == lab
        ax.scatter(X_pca_2d[m_, 0], X_pca_2d[m_, 1], s=3, alpha=0.4, label=lab,
                   color=palette.get(lab, "#888888"))
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_title(title)
    ax.legend(markerscale=4, fontsize=9); ax.grid(alpha=0.2)
plt.suptitle("PCA biplot — source vs biological signal (PCA fitted on healthy-train)", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(batch_dir, "pca_biplot_source_vs_tumour.png"), dpi=150)
plt.close()
log("2A: PCA biplot saved.")

[HASEM] Step 2: provenance / batch-structure analysis...
[HASEM] 2A: PCA biplot saved.


### 2B — GTEx vs TCGA/TARGET-normal separability

In [20]:
src_bin = (meta["source"].values == "TCGA/TARGET").astype(int)
y_tr, y_ho = src_bin[healthy_train_mask], src_bin[healthy_holdout_mask]
n_tcga_tr, n_tcga_ho = int(y_tr.sum()), int(y_ho.sum())
log(f"2B: healthy-train: {len(y_tr)-n_tcga_tr} GTEx / {n_tcga_tr} TCGA-TARGET normal | "
    f"healthy-holdout: {len(y_ho)-n_tcga_ho} GTEx / {n_tcga_ho} TCGA-TARGET normal")
def _src_auc(Xmat):
    if min(n_tcga_tr, len(y_tr) - n_tcga_tr) >= 5 and min(n_tcga_ho, len(y_ho) - n_tcga_ho) >= 3:
        clf = LogisticRegression(max_iter=1000, solver="saga", random_state=RANDOM_SEED)
        clf.fit(Xmat[healthy_train_mask], y_tr)
        return roc_auc_score(y_ho, clf.predict_proba(Xmat[healthy_holdout_mask])[:, 1])
    return float("nan")
source_auc = _src_auc(X_scaled.values)                                   # after any correction (equals uncorrected if mode="none")
source_auc_uncorrected = _src_auc(X_scaled_original.values)
batch_auc = source_auc
log(f"2B: GTEx vs TCGA/TARGET-normal separability (train -> holdout): uncorrected ROC-AUC = {source_auc_uncorrected:.4f} | "
    f"after correction (mode={COMBAT_MODE}) = {source_auc:.4f}")

pd.DataFrame([{"Check": "GTEx vs TCGA/TARGET normal (healthy only; train -> holdout)",
               "ROC_AUC": source_auc, "ROC_AUC_uncorrected": source_auc_uncorrected, "ComBat_mode": COMBAT_MODE,
               "n_train": int(len(y_tr)), "n_holdout": int(len(y_ho))}]
    ).to_csv(os.path.join(batch_dir, "true_batch_negative_control.csv"), index=False)
pd.DataFrame({"Check": ["Source separability within healthy class, uncorrected (ROC-AUC)", "Source separability after correction (ROC-AUC)", "Correction mode"],
              "Result": [f"{source_auc_uncorrected:.4f}", f"{source_auc:.4f}", COMBAT_MODE]}
    ).to_csv(os.path.join(batch_dir, "batch_effect_summary.csv"), index=False)
log("Step 2 complete.")

[HASEM] 2B: healthy-train: 5688 GTEx / 0 TCGA-TARGET normal | healthy-holdout: 1273 GTEx / 0 TCGA-TARGET normal
[HASEM] 2B: GTEx vs TCGA/TARGET-normal separability (train -> holdout): uncorrected ROC-AUC = nan | after correction (mode=none) = nan
[HASEM] Step 2 complete.


## Cell 3 — T² and Q scoring, control limits and evaluation

Limits come from healthy calibration samples; evaluation uses the untouched pool (healthy holdout + tumours).

In [21]:
log("Step 3: T2 and Q-residual scores, thresholds from the calibration partition...")

X_arr = X_scaled.values
X_pca_arr = pca_unsup.transform(X_arr)
X_recon   = pca_unsup.inverse_transform(X_pca_arr)
residuals = X_arr - X_recon
safe_eigen = np.where(pca_unsup.explained_variance_ <= 0, 1e-12, pca_unsup.explained_variance_)
T2_scores = np.sum((X_pca_arr ** 2) / safe_eigen, axis=1)
Q_scores  = np.sum(residuals ** 2, axis=1)

m = int(healthy_train_mask.sum())     # samples the PCA was fitted on
a = n_comp

[HASEM] Step 3: T2 and Q-residual scores, thresholds from the calibration partition...


### 3b — Empirical and theoretical control limits

In [22]:
T2_thresh_emp = np.nanpercentile(T2_scores[healthy_calib_mask], THRESH_PCT)
Q_thresh_emp  = np.nanpercentile(Q_scores[healthy_calib_mask],  THRESH_PCT)
log(f"  Empirical limits ({THRESH_PCT}th pct of calibration set, n={int(healthy_calib_mask.sum())}): "
    f"T2={T2_thresh_emp:.3f}  Q={Q_thresh_emp:.3f}")

if m > a:
    Fcrit = f_dist.ppf(1 - ANOMALY_ALPHA, a, m - a)
    T2_thresh_theo = (a * (m - 1) * (m + 1)) / (m * (m - a)) * Fcrit
else:
    T2_thresh_theo = T2_thresh_emp
    log("  Warning: not enough training samples for theoretical T2 — using empirical.")
try:
    resid_var = np.var(residuals[healthy_train_mask], axis=0)
    th1, th2, th3 = resid_var.sum(), (resid_var**2).sum(), (resid_var**3).sum()
    h0 = 1 - (2*th1*th3) / (3*th2**2 + 1e-12)
    if h0 <= 0 or th2 <= 0:
        raise ValueError("Unstable Q parameters")
    ca = np.sqrt(2*th2*(h0**2)) / (th1 + 1e-12)
    fa_ = f_dist.ppf(1 - ANOMALY_ALPHA, max(1, round(h0*2)), max(1, round(h0*2)))
    Q_thresh_theo = th1 * ((np.sqrt(fa_)*ca + 1 + th2*h0*(h0-1)/th1**2) ** (1.0/h0))
except Exception as e:
    Q_thresh_theo = Q_thresh_emp
    log(f"  Q-theoretical fallback to empirical: {e}")
log(f"  Theoretical limits: T2={T2_thresh_theo:.3f}  Q={Q_thresh_theo:.3f}")

[HASEM]   Empirical limits (99.0th pct of calibration set, n=1188): T2=1645.288  Q=1329.645
[HASEM]   Theoretical limits: T2=1921.944  Q=783.461


### 3c — Flags and scores

In [23]:
y_pred_emp  = ((T2_scores > T2_thresh_emp)  | (Q_scores > Q_thresh_emp )).astype(int)
y_pred_theo = ((T2_scores > T2_thresh_theo) | (Q_scores > Q_thresh_theo)).astype(int)
y_pred_Q    = (Q_scores > Q_thresh_emp).astype(int)

# z-normalisation uses the CALIBRATION set (unseen by the PCA fit)
T2_z = (T2_scores - T2_scores[healthy_calib_mask].mean()) / (T2_scores[healthy_calib_mask].std() + 1e-12)
Q_z  = (Q_scores  - Q_scores[healthy_calib_mask].mean())  / (Q_scores[healthy_calib_mask].std()  + 1e-12)
combined_score = (T2_z + Q_z) / 2.0

### 3d — Isolation Forest benchmark

Fitted on healthy train only.

In [24]:
iso = IsolationForest(random_state=RANDOM_SEED, contamination="auto", n_jobs=-1)
iso.fit(X_arr[healthy_train_mask])
iso_scores = -iso.decision_function(X_arr)
iso_preds  = (iso.predict(X_arr) == -1).astype(int)

### 3e — Evaluation helper and headline results

Percentages with Wilson confidence intervals.

In [25]:
def evaluate(y_true, y_pred, score, label):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    roc = roc_auc_score(y_true, score)
    pr  = average_precision_score(y_true, score)
    det = 100 * tp / (tp + fn + 1e-12)
    fal = 100 * fp / (fp + tn + 1e-12)
    d_lo, d_hi = wilson_ci(tp, tp + fn)
    f_lo, f_hi = wilson_ci(fp, fp + tn)
    print(f"  {label:30s}  ROC={roc:.4f}  PR={pr:.4f}  Det={det:.2f}%  FA={fal:.2f}%")
    return dict(label=label, roc=roc, pr=pr, detect=det, false_alarm=fal,
                detect_ci_lo=d_lo, detect_ci_hi=d_hi, fa_ci_lo=f_lo, fa_ci_hi=f_hi,
                tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))

quick_metrics_cache = {}
def quick_metrics(y_true, y_pred, score):
    r = evaluate(y_true, y_pred, score, "_")
    return r["roc"], r["pr"], r["detect"], r["false_alarm"]

ev   = eval_mask
y_ev = y_binary.values[ev]
print(f"\n[HASEM] LEAKAGE-SAFE unsupervised results — evaluation pool only "
      f"(n={int(ev.sum()):,}: {int((y_ev==0).sum()):,} healthy-holdout, {int((y_ev==1).sum()):,} tumours):")
res_emp  = evaluate(y_ev, y_pred_emp[ev],  combined_score[ev], "T²+Q Empirical")
res_theo = evaluate(y_ev, y_pred_theo[ev], combined_score[ev], "T²+Q Theoretical")
res_iso  = evaluate(y_ev, iso_preds[ev],   iso_scores[ev],     "Isolation Forest")
pd.DataFrame([res_emp, res_theo, res_iso]).to_csv(
    os.path.join(OUTDIR, "unsupervised", "unsup_metrics.csv"), index=False)


[HASEM] LEAKAGE-SAFE unsupervised results — evaluation pool only (n=9,775: 1,273 healthy-holdout, 8,502 tumours):
  T²+Q Empirical                  ROC=0.9521  PR=0.9923  Det=70.88%  FA=2.44%
  T²+Q Theoretical                ROC=0.9521  PR=0.9923  Det=86.18%  FA=10.92%
  Isolation Forest                ROC=0.8251  PR=0.9443  Det=34.25%  FA=8.96%


### 3f — Statistic ablation

In [26]:
print("\n[HASEM] Statistic ablation (evaluation pool):")
score_and = np.minimum(T2_scores / T2_thresh_emp, Q_scores / Q_thresh_emp)
abl = [
    evaluate(y_ev, y_pred_Q[ev], Q_scores[ev], "Q only"),
    evaluate(y_ev, (T2_scores[ev] > T2_thresh_emp).astype(int), T2_scores[ev], "T2 only"),
    evaluate(y_ev, y_pred_emp[ev], combined_score[ev], "T2 OR Q"),
    evaluate(y_ev, ((T2_scores[ev] > T2_thresh_emp) & (Q_scores[ev] > Q_thresh_emp)).astype(int),
             score_and[ev], "T2 AND Q"),
]
abl_df = pd.DataFrame(abl)
abl_df.to_csv(os.path.join(OUTDIR, "unsupervised", "ablation_all.csv"), index=False)
for r in abl:
    tag = r["label"].replace(" ", "_").replace("²", "2")
    pd.DataFrame([r]).to_csv(os.path.join(OUTDIR, "unsupervised", f"ablation_{tag}.csv"), index=False)


[HASEM] Statistic ablation (evaluation pool):
  Q only                          ROC=0.9380  PR=0.9895  Det=54.56%  FA=1.65%
  T2 only                         ROC=0.9601  PR=0.9937  Det=70.85%  FA=1.41%
  T2 OR Q                         ROC=0.9521  PR=0.9923  Det=70.88%  FA=2.44%
  T2 AND Q                        ROC=0.9442  PR=0.9911  Det=54.54%  FA=0.63%


### 3g — False alarms by partition

In [27]:
rows = []
for name, msk in [("healthy_train (used to FIT; in-sample)", healthy_train_mask),
                  ("healthy_calib (used to set limits)",      healthy_calib_mask),
                  ("healthy_holdout (honest estimate)",       healthy_holdout_mask)]:
    k_or, k_q, n = int(y_pred_emp[msk].sum()), int(y_pred_Q[msk].sum()), int(msk.sum())
    rows.append({"Partition": name, "N": n,
                 "FA_OR_pct": round(100*k_or/n, 2), "FA_Qonly_pct": round(100*k_q/n, 2)})
fa_part = pd.DataFrame(rows)
fa_part.to_csv(os.path.join(OUTDIR, "unsupervised", "false_alarm_by_partition.csv"), index=False)
print("\n[HASEM] False alarms by partition:"); print(fa_part.to_string(index=False))


[HASEM] False alarms by partition:
                             Partition    N  FA_OR_pct  FA_Qonly_pct
healthy_train (used to FIT; in-sample) 5688      20.34          0.00
    healthy_calib (used to set limits) 1188       1.60          1.01
     healthy_holdout (honest estimate) 1273       2.44          1.65


### 3h — False alarms by source (healthy holdout only)

In [28]:
rows = []

for grp in ["GTEx", "TCGA", "TARGET"]:
    sel = healthy_holdout_mask & (meta["source"].values == grp)
    n = int(sel.sum())

    if n == 0:
        continue

    k_or = int(y_pred_emp[sel].sum())
    k_q = int(y_pred_Q[sel].sum())

    lo_or, hi_or = wilson_ci(k_or, n)
    lo_q, hi_q = wilson_ci(k_q, n)

    rows.append({
        "Group": f"{grp} healthy (holdout)",
        "N": n,
        "FA_OR_pct": round(100 * k_or / n, 2),
        "FA_OR_CI_lo": lo_or,
        "FA_OR_CI_hi": hi_or,
        "FA_Qonly_pct": round(100 * k_q / n, 2),
        "FA_Q_CI_lo": lo_q,
        "FA_Q_CI_hi": hi_q
    })

fa_src = pd.DataFrame(rows)
fa_src.to_csv(
    os.path.join(batch_dir, "false_alarm_by_source.csv"),
    index=False
)

print("\n[HASEM] False alarms by source (healthy holdout only):")
print(fa_src.to_string(index=False))


[HASEM] False alarms by source (healthy holdout only):
                 Group    N  FA_OR_pct  FA_OR_CI_lo  FA_OR_CI_hi  FA_Qonly_pct  FA_Q_CI_lo  FA_Q_CI_hi
GTEx healthy (holdout) 1161       2.07         1.39         3.06          1.72        1.12        2.65
TCGA healthy (holdout)  112       6.25         3.06        12.34          0.89        0.16        4.88


### 3i — T² vs Q scatter (evaluation pool)

In [29]:
fig, ax = plt.subplots(figsize=(8, 6))
for lab, grp in pd.DataFrame({"T2": T2_scores[ev], "Q": Q_scores[ev], "tumor": y_ev}).groupby("tumor"):
    ax.scatter(grp["T2"], grp["Q"], s=4, alpha=0.3, color={0: "#4CAF50", 1: "#F44336"}[lab],
               label="Healthy (holdout)" if lab == 0 else "Tumour")
ax.axvline(T2_thresh_emp, color="#1F4E79", ls="--", lw=1.2, label="T² limit")
ax.axhline(Q_thresh_emp,  color="#FF6600", ls="--", lw=1.2, label="Q limit")
ax.set_xlabel("Hotelling T²"); ax.set_ylabel("Q-Residual")
ax.set_title("MSPC anomaly map — evaluation pool"); ax.legend(markerscale=4)
ax.set_xlim(left=0); ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "unsupervised", "T2_Q_scatter.png"), dpi=150)
plt.close()
log("Step 3 complete.")

[HASEM] Step 3 complete.


In [30]:
# Optional final cell: archive per-sample scores. Does not change any result.
# Run in the same session after Cell 3 (variables below are created there).
assert len(meta) == len(T2_scores) == len(Q_scores) == len(iso_scores), "row alignment check failed"
per_sample = pd.DataFrame({
    "sample": meta.index,
    "partition": meta["partition"].values,
    "source": meta["source"].values,
    "T2": T2_scores,
    "Q": Q_scores,
    "flag_T2_or_Q_empirical": y_pred_emp,
    "flag_T2_or_Q_theoretical": y_pred_theo,
    "flag_Q_only_empirical": y_pred_Q,
    "isolation_forest_score": iso_scores,
    "isolation_forest_flag": iso_preds,
})
per_sample.to_csv(os.path.join(OUTDIR, "unsupervised", "per_sample_scores.csv"), index=False)
print("per-sample scores written:", per_sample.shape)

per-sample scores written: (17330, 10)


## Cell 4 — Per-cancer detection rates

Cancer type, sample size, detection rate (%) and 95% Wilson interval.

In [31]:
log("Step 4: Computing full per-cancer detection rates...")

def full_per_cancer_table(y_true_arr, y_pred_arr, cancer_labels, tag=""):
    rows = []
    mask_tumor = (y_true_arr == 1)
    df = pd.DataFrame({
        "cancer": cancer_labels[mask_tumor],
        "detected": y_pred_arr[mask_tumor]
    })
    for cancer, grp in df.groupby("cancer"):
        n   = len(grp)
        k   = grp["detected"].sum()
        lo, hi = wilson_ci(k, n)
        rows.append({
            "Cancer Type": cancer,
            "N Samples": n,
            "Detected": int(k),
            "Detection Rate (%)": round(100*k/n, 2) if n > 0 else np.nan,
            "95% CI Lower": lo,
            "95% CI Upper": hi
        })
    result = pd.DataFrame(rows).sort_values("Detection Rate (%)", ascending=False)
    result.to_csv(
        os.path.join(OUTDIR, "per_cancer", f"per_cancer_{tag}.csv"), index=False)
    return result

# Only evaluation-pool tumours are counted (others marked -1); none was used to fit or calibrate.
y_eval_arr = np.where(eval_mask, y_binary.values, -1)
pc_emp  = full_per_cancer_table(y_eval_arr, y_pred_emp,  y_detailed.values, "empirical")
pc_theo = full_per_cancer_table(y_eval_arr, y_pred_theo, y_detailed.values, "theoretical")
pc_Q    = full_per_cancer_table(y_eval_arr, y_pred_Q,    y_detailed.values, "Qonly")

print("\n[HASEM] Per-cancer detection rates (Empirical threshold) — all types:")
print(pc_emp.to_string(index=False))

[HASEM] Step 4: Computing full per-cancer detection rates...

[HASEM] Per-cancer detection rates (Empirical threshold) — all types:
            Cancer Type  N Samples  Detected  Detection Rate (%)  95% CI Lower  95% CI Upper
                   Skin        102       101               99.02         94.65         99.83
                    Eye         79        78               98.73         93.17         99.78
          Adrenal gland         77        76               98.70         93.00         99.77
       Lymphatic tissue         47        46               97.87         88.89         99.62
       Soft tissue,Bone        256       249               97.27         94.46         98.67
                 Thymus        117       113               96.58         91.54         98.66
            Paraganglia        174       161               92.53         87.64         95.58
                  Liver        319       295               92.48         89.05         94.89
                 Cervix        

### 4b — Bar chart

In [32]:
plot_df = pc_emp[pc_emp["Cancer Type"] != "HEALTHY"].copy()
fig, ax = plt.subplots(figsize=(10, max(6, len(plot_df)*0.35)))
colors = ["#1F4E79" if r >= 90 else "#2196F3" if r >= 70
          else "#FF9800" if r >= 50 else "#F44336"
          for r in plot_df["Detection Rate (%)"]]
bars = ax.barh(plot_df["Cancer Type"], plot_df["Detection Rate (%)"],
               color=colors, edgecolor="white", height=0.7)
ax.errorbar(
    plot_df["Detection Rate (%)"],
    range(len(plot_df)),
    xerr=[plot_df["Detection Rate (%)"] - plot_df["95% CI Lower"],
          plot_df["95% CI Upper"]    - plot_df["Detection Rate (%)"]],
    fmt="none", color="black", capsize=3, lw=1
)
ax.axvline(90, color="green", ls="--", lw=1, label="90% line")
ax.set_xlabel("Detection Rate (%)"); ax.set_xlim(0, 105)
ax.set_title("Per-Cancer Detection Rates with 95% CI\n(HASEM T²+Q Empirical Threshold)")
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,"per_cancer","per_cancer_detection_rates.png"),
            dpi=150, bbox_inches="tight")
plt.close()

log(f"Step 4 complete. Results for {len(pc_emp)} cancer types saved.")

[HASEM] Step 4 complete. Results for 28 cancer types saved.


## Cell 4B — Cell-line sensitivity

Cell lines are never used for fitting or calibration; this cell shows headline numbers with and without them.

In [33]:
log("Step 4B: cell-line sensitivity check...")
cellline_mask = (meta["_sample_type"] == "Cell Line").values
log(f"  Cell-line samples in the dataset: {int(cellline_mask.sum())} "
    f"(in evaluation pool: {int((cellline_mask & eval_mask).sum())})")

ev_nocl = eval_mask & ~cellline_mask

def _summ(msk, label):
    y = y_binary.values[msk]
    out = {}
    for rule, pred, sc in [("Q only", y_pred_Q, Q_scores), ("T2 OR Q", y_pred_emp, combined_score)]:
        r = evaluate(y, pred[msk], sc[msk], f"{label} | {rule}")
        out[rule] = r
    return out

print("\n[HASEM] With and without cell lines (evaluation pool):")
with_cl = _summ(eval_mask, "incl. cell lines")
no_cl   = _summ(ev_nocl,   "patient tumours only")
rows = []
for rule in ["Q only", "T2 OR Q"]:
    for label, r in [("incl. cell lines", with_cl[rule]), ("patient tumours only", no_cl[rule])]:
        rows.append({"Rule": rule, "Evaluation set": label, "ROC": r["roc"], "PR": r["pr"],
                     "Detection_pct": r["detect"], "FalseAlarm_pct": r["false_alarm"]})
pd.DataFrame(rows).to_csv(os.path.join(OUTDIR, "unsupervised", "celline_sensitivity_comparison.csv"), index=False)

if cellline_mask.sum() > 0:
    flagged = int(y_pred_Q[cellline_mask].sum())
    log(f"  Cell lines flagged by Q alone: {flagged}/{int(cellline_mask.sum())} "
        f"({100*flagged/cellline_mask.sum():.2f}%)")

y_nc_arr = np.where(ev_nocl, y_binary.values, -1)
df_percancer_patient = full_per_cancer_table(y_nc_arr, y_pred_emp, y_detailed.values, tag="patientonly")
log("Step 4B complete.")

[HASEM] Step 4B: cell-line sensitivity check...
[HASEM]   Cell-line samples in the dataset: 0 (in evaluation pool: 0)

[HASEM] With and without cell lines (evaluation pool):
  incl. cell lines | Q only       ROC=0.9380  PR=0.9895  Det=54.56%  FA=1.65%
  incl. cell lines | T2 OR Q      ROC=0.9521  PR=0.9923  Det=70.88%  FA=2.44%
  patient tumours only | Q only   ROC=0.9380  PR=0.9895  Det=54.56%  FA=1.65%
  patient tumours only | T2 OR Q  ROC=0.9521  PR=0.9923  Det=70.88%  FA=2.44%
[HASEM] Step 4B complete.


## Cell 5 — Leave-one-cancer-out (LOCO) validation

**LOCO-A:** global model, per-cancer evaluation. **LOCO-B:** reference refitted on healthy train with limits from healthy calibration.

In [34]:
log("Step 5: LOCO validation...")
loco_dir = os.path.join(OUTDIR, "loco")
cancers = sorted(y_detailed[y_binary == 1].dropna().unique())

log("Running LOCO-A (global PCA, per-cancer evaluation)...")
loco_a_rows = []
for cancer in cancers:
    mask = (((y_detailed == cancer) & (y_binary == 1)).values) & eval_mask
    n = int(mask.sum())
    if n < MIN_CANCER_SAMPLES:
        continue
    Xc_pca = pca_unsup.transform(X_arr[mask])
    T2c = compute_T2(Xc_pca, pca_unsup.explained_variance_)
    Qc  = np.sum((X_arr[mask] - pca_unsup.inverse_transform(Xc_pca))**2, axis=1)
    det = int(((T2c > T2_thresh_emp) | (Qc > Q_thresh_emp)).sum())
    lo, hi = wilson_ci(det, n)
    loco_a_rows.append(dict(Cancer=cancer, N=n, Detected=det,
                            DetectRate=round(100*det/n, 2), CI_Lower=lo, CI_Upper=hi))
df_locoA = pd.DataFrame(loco_a_rows).sort_values("DetectRate", ascending=False)
df_locoA.to_csv(os.path.join(loco_dir, "LOCO_A_results.csv"), index=False)
log(f"LOCO-A complete: {len(df_locoA)} cancer types evaluated.")

def fit_reference(train_rows, calib_rows):
    Xref = X_log.values[train_rows]
    mu = Xref.mean(0); sd = Xref.std(0, ddof=1); sd[sd == 0] = 1
    pca_t = PCA(n_components=VAR_EXPLAINED, svd_solver="full", random_state=RANDOM_SEED)
    pca_t.fit((Xref - mu) / sd)
    Xc = (X_log.values[calib_rows] - mu) / sd
    Zc = pca_t.transform(Xc)
    T2c = compute_T2(Zc, pca_t.explained_variance_)
    Qc  = np.sum((Xc - pca_t.inverse_transform(Zc))**2, axis=1)
    return dict(mu=mu, sd=sd, pca=pca_t,
                T2_lim=np.nanpercentile(T2c, THRESH_PCT), Q_lim=np.nanpercentile(Qc, THRESH_PCT))

def detect_with(ref, test_rows):
    Xt = (X_log.values[test_rows] - ref["mu"]) / ref["sd"]
    Zt = ref["pca"].transform(Xt)
    T2t = compute_T2(Zt, ref["pca"].explained_variance_)
    Qt  = np.sum((Xt - ref["pca"].inverse_transform(Zt))**2, axis=1)
    return int(((T2t > ref["T2_lim"]) | (Qt > ref["Q_lim"])).sum())

[HASEM] Step 5: LOCO validation...
[HASEM] Running LOCO-A (global PCA, per-cancer evaluation)...
[HASEM] LOCO-A complete: 28 cancer types evaluated.


### LOCO-B

In [35]:
log("Running LOCO-B (re-fitted reference)...")
ref_B = None if LOCO_B_REFIT_EACH else fit_reference(healthy_train_mask, healthy_calib_mask)
loco_b_rows = []
for cancer in cancers:
    test_mask = (((y_detailed == cancer) & (y_binary == 1)).values) & eval_mask
    n_test = int(test_mask.sum())
    if n_test < MIN_CANCER_SAMPLES:
        continue
    ref = ref_B if ref_B is not None else fit_reference(healthy_train_mask, healthy_calib_mask)
    det = detect_with(ref, test_mask)
    lo, hi = wilson_ci(det, n_test)
    loco_b_rows.append(dict(Cancer=cancer, N=n_test, Detected=det,
                            DetectRate=round(100*det/n_test, 2), CI_Lower=lo, CI_Upper=hi))
df_locoB = pd.DataFrame(loco_b_rows).sort_values("DetectRate", ascending=False)
df_locoB.to_csv(os.path.join(loco_dir, "LOCO_B_results.csv"), index=False)
log(f"LOCO-B complete: {len(df_locoB)} cancer types evaluated.")

[HASEM] Running LOCO-B (re-fitted reference)...
[HASEM] LOCO-B complete: 28 cancer types evaluated.


### Combined LOCO comparison plot

In [36]:
merged = df_locoA.merge(df_locoB, on="Cancer", suffixes=("_A", "_B"))
fig, ax = plt.subplots(figsize=(10, max(6, len(merged)*0.35)))
y_pos = range(len(merged))
ax.barh([p+0.2 for p in y_pos], merged["DetectRate_A"], height=0.35, color="#1F4E79", label="LOCO-A (global PCA)")
ax.barh([p-0.2 for p in y_pos], merged["DetectRate_B"], height=0.35, color="#2196F3", label="LOCO-B (re-fitted reference)")
ax.set_yticks(list(y_pos)); ax.set_yticklabels(merged["Cancer"], fontsize=8)
ax.set_xlabel("Detection Rate (%)"); ax.set_xlim(0, 110)
ax.set_title("LOCO validation: LOCO-A vs LOCO-B per cancer type")
ax.axvline(90, color="red", ls="--", lw=1, label="90% reference")
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(loco_dir, "LOCO_comparison.png"), dpi=150, bbox_inches="tight")
plt.close()
log("Step 5 complete.")

[HASEM] Step 5 complete.


## Cell 5C — True LOCO (organ never seen)

Tissue-matched healthy samples are removed from both the PCA reference and the calibration set before that cancer is scored.

In [37]:
log("Step 5C: true LOCO (tissue-matched exclusion)...")
tissue_all = meta["_primary_site"].values
results, skipped = [], []

for cancer in cancers:
    test_mask = (((y_detailed == cancer) & (y_binary == 1)).values) & eval_mask
    n_test = int(test_mask.sum())
    if n_test < MIN_CANCER_SAMPLES:
        continue
    same_tissue = (tissue_all == cancer)
    n_removed   = int(((healthy_train_mask | healthy_calib_mask) & same_tissue).sum())
    tr_rows = healthy_train_mask & ~same_tissue
    ca_rows = healthy_calib_mask & ~same_tissue
    if n_removed == 0:
        skipped.append((cancer, "no matching healthy tissue in the reference — LOCO-B result applies unchanged"))
        continue
    if tr_rows.sum() < 30 or ca_rows.sum() < 30:
        skipped.append((cancer, f"too few healthy samples left (train {int(tr_rows.sum())}, calib {int(ca_rows.sum())})"))
        continue
    ref = fit_reference(tr_rows, ca_rows)
    det = detect_with(ref, test_mask)
    lo, hi = wilson_ci(det, n_test)
    results.append(dict(Cancer=cancer, N_test=n_test, N_healthy_removed=n_removed,
                        N_train_remaining=int(tr_rows.sum()), N_calib_remaining=int(ca_rows.sum()),
                        Detected=det, DetectRate=round(100*det/n_test, 2), CI_Lower=lo, CI_Upper=hi))

df_true_loco = pd.DataFrame(results).sort_values("DetectRate", ascending=False)
df_true_loco.to_csv(os.path.join(OUTDIR, "loco", "LOCO_C_tissue_excluded.csv"), index=False)
log(f"Step 5C complete: {len(df_true_loco)} cancer types tested with matching tissue fully removed.")
if skipped:
    log(f"  {len(skipped)} cancer types skipped or unchanged:")
    for c, reason in skipped:
        log(f"    {c}: {reason}")

[HASEM] Step 5C: true LOCO (tissue-matched exclusion)...
[HASEM] Step 5C complete: 24 cancer types tested with matching tissue fully removed.
[HASEM]   4 cancer types skipped or unchanged:
[HASEM]     Adrenal gland: no matching healthy tissue in the reference — LOCO-B result applies unchanged
[HASEM]     Eye: no matching healthy tissue in the reference — LOCO-B result applies unchanged
[HASEM]     Lining of body cavities: no matching healthy tissue in the reference — LOCO-B result applies unchanged
[HASEM]     Lymphatic tissue: no matching healthy tissue in the reference — LOCO-B result applies unchanged


## Cell 6 — Supervised classification with patient-group splits

**6a.** Patient-group split and PCA for the supervised models.

In [38]:
log("Step 6: supervised classification and calibration...")

from sklearn.model_selection import train_test_split

part = meta["partition"].astype(str).to_numpy()
group = meta["patient_group"].astype(str).to_numpy()
y_all = y_binary.to_numpy(dtype=int)
site_all = y_detailed.fillna("Unknown").astype(str).to_numpy()

eligible = (
    np.isin(part, ["healthy_train", "healthy_calib", "healthy_holdout", "tumour"])
    & pd.notna(meta["patient_group"]).to_numpy()
)
all_idx = np.flatnonzero(eligible)

healthy_test_idx = np.flatnonzero(
    eligible & (part == "healthy_holdout")
)
tumour_idx = np.flatnonzero(
    eligible & eval_tumour_mask
)

# Each tumour patient group must have one cancer label for stratification.
tumour_groups = pd.DataFrame({
    "idx": tumour_idx,
    "group": group[tumour_idx],
    "site": site_all[tumour_idx]
})
group_sites = tumour_groups.groupby("group")["site"].agg(
    lambda x: x.value_counts().index[0]
)
site_counts = group_sites.value_counts()
strata = group_sites.where(group_sites.map(site_counts) >= 2, "Rare")

# Split tumour PATIENT GROUPS, not individual samples.
tumour_group_ids = group_sites.index.to_numpy()
_, tumour_test_groups = train_test_split(
    tumour_group_ids,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=strata.to_numpy()
)

tumour_test_idx = tumour_idx[
    np.isin(group[tumour_idx], tumour_test_groups)
]
sup_test_idx = np.unique(np.concatenate([
    healthy_test_idx, tumour_test_idx
]))

test_groups = set(group[sup_test_idx])
remaining_idx = np.array([
    i for i in all_idx if group[i] not in test_groups
], dtype=int)

# A group must have one binary label. Stop if that assumption is violated.
group_labels = pd.DataFrame({
    "group": group[remaining_idx],
    "label": y_all[remaining_idx]
}).groupby("group")["label"].nunique()

if (group_labels > 1).any():
    raise ValueError(
        "Some remaining patient groups have both labels. "
        "Inspect patient_group assignments before proceeding."
    )

remaining_group_labels = pd.DataFrame({
    "group": group[remaining_idx],
    "label": y_all[remaining_idx]
}).drop_duplicates("group").set_index("group")["label"]

fit_groups, cal_groups = train_test_split(
    remaining_group_labels.index.to_numpy(),
    test_size=0.15,
    random_state=RANDOM_SEED,
    stratify=remaining_group_labels.to_numpy()
)

sup_fit_idx = remaining_idx[np.isin(group[remaining_idx], fit_groups)]
sup_cal_idx = remaining_idx[np.isin(group[remaining_idx], cal_groups)]

fit_set, cal_set, test_set = (
    set(group[sup_fit_idx]),
    set(group[sup_cal_idx]),
    set(group[sup_test_idx])
)
assert not (fit_set & cal_set), "Patient overlap: fit/calibration"
assert not (fit_set & test_set), "Patient overlap: fit/test"
assert not (cal_set & test_set), "Patient overlap: calibration/test"

log(
    f"  Group-aware split: fit {len(sup_fit_idx):,} | "
    f"calibration {len(sup_cal_idx):,} | test {len(sup_test_idx):,}"
)
print("Patient-group overlap counts:",
      len(fit_set & cal_set),
      len(fit_set & test_set),
      len(cal_set & test_set))

roles = np.full(len(meta), "unused", dtype=object)
roles[sup_fit_idx] = "fit"
roles[sup_cal_idx] = "calibration"
roles[sup_test_idx] = "test"
pd.DataFrame({
    "sample": meta.index,
    "patient_group": group,
    "role": roles,
    "source": meta["source"].to_numpy(),
    "partition": part,
    "label": y_all
}).to_csv(
    os.path.join(OUTDIR, "models", "supervised_split_manifest.csv"),
    index=False
)

# PCA is fitted on supervised fit rows only.
pca_sup = PCA(
    n_components=n_comp,
    svd_solver="full",
    random_state=RANDOM_SEED
)
pca_sup.fit(X_scaled.values[sup_fit_idx])

X_tr_pca = pca_sup.transform(X_scaled.values[sup_fit_idx])
y_tr = y_all[sup_fit_idx]
X_cal_pca = pca_sup.transform(X_scaled.values[sup_cal_idx])
y_cal = y_all[sup_cal_idx]
X_te_pca = pca_sup.transform(X_scaled.values[sup_test_idx])
y_te = y_all[sup_test_idx]

[HASEM] Step 6: supervised classification and calibration...
[HASEM]   Group-aware split: fit 11,554 | calibration 2,123 | test 2,974
Patient-group overlap counts: 0 0 0


### 6b — Train classifiers

In [39]:
def train_eval(clf, Xtr, ytr, Xte, yte, name):
    clf.fit(Xtr, ytr)
    proba = clf.predict_proba(Xte)[:, 1]
    pred  = (proba > 0.5).astype(int)
    roc, pr = roc_auc_score(yte, proba), average_precision_score(yte, proba)
    acc, f1, rec = accuracy_score(yte, pred), f1_score(yte, pred), recall_score(yte, pred)
    brier = brier_score_loss(yte, proba)
    cm = confusion_matrix(yte, pred)
    print(f"  {name:22s}  ROC={roc:.4f}  PR={pr:.4f}  Acc={acc:.4f}  F1={f1:.4f}  Rec={rec:.4f}  Brier={brier:.4f}")
    return dict(model=clf, proba=proba, pred=pred, name=name,
                roc=roc, pr=pr, acc=acc, f1=f1, rec=rec, brier=brier, cm=cm)

print("\n[HASEM] Binary supervised classification results (untouched test set):")
lr  = LogisticRegression(max_iter=10000, solver="saga", class_weight="balanced", random_state=RANDOM_SEED)
rf  = RandomForestClassifier(n_estimators=200, n_jobs=-1, class_weight="balanced", random_state=RANDOM_SEED)
res_lr = train_eval(lr, X_tr_pca, y_tr, X_te_pca, y_te, "Logistic Regression")
res_rf = train_eval(rf, X_tr_pca, y_tr, X_te_pca, y_te, "Random Forest")
res_xgb = None
if HAVE_XGB:
    xgb_clf = xgb.XGBClassifier(n_estimators=200, eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=-1)
    res_xgb = train_eval(xgb_clf, X_tr_pca, y_tr, X_te_pca, y_te, "XGBoost")


[HASEM] Binary supervised classification results (untouched test set):
  Logistic Regression     ROC=0.9996  PR=0.9996  Acc=0.9963  F1=0.9968  Rec=0.9953  Brier=0.0033
  Random Forest           ROC=0.9991  PR=0.9993  Acc=0.9909  F1=0.9921  Rec=0.9924  Brier=0.0383
  XGBoost                 ROC=0.9995  PR=0.9996  Acc=0.9936  F1=0.9944  Rec=0.9935  Brier=0.0053


### 6c — ROC and PR curves

In [40]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for res in [res_lr, res_rf, res_xgb]:
    if res is None: continue
    RocCurveDisplay.from_predictions(y_te, res["proba"], name=f"{res['name']} (AUC={res['roc']:.4f})", ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_te, res["proba"], name=f"{res['name']} (AP={res['pr']:.4f})", ax=axes[1])
axes[0].set_title("ROC curves"); axes[1].set_title("Precision-Recall curves")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "supervised", "roc_pr_curves.png"), dpi=150)
plt.close()

### 6d — Calibration

Platt scaling fitted on the calibration rows and scored on the test set.

In [41]:
log("Step 6d: calibration curves (Platt scaling fitted on the calibration split)...")
def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)
platt_models = {}
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
models_to_cal = [r for r in [res_lr, res_rf, res_xgb] if r is not None]
for ax, res in zip(axes, models_to_cal):
    p_cal = res["model"].predict_proba(X_cal_pca)[:, 1]
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(_logit(p_cal), y_cal)
    platt_models[res["name"]] = platt
    cal_proba = platt.predict_proba(_logit(res["proba"]))[:, 1]
    frac_pos, mean_pred = calibration_curve(y_te, res["proba"], n_bins=10)
    frac_pos_cal, mean_pred_cal = calibration_curve(y_te, cal_proba, n_bins=10)
    brier_raw, brier_cal = brier_score_loss(y_te, res["proba"]), brier_score_loss(y_te, cal_proba)
    res["brier_platt"] = brier_cal
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
    ax.plot(mean_pred, frac_pos, "o-", color="#F44336", label=f"Uncalibrated (Brier={brier_raw:.4f})")
    ax.plot(mean_pred_cal, frac_pos_cal, "s-", color="#1F4E79", label=f"Platt-scaled (Brier={brier_cal:.4f})")
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Fraction of positives")
    ax.set_title(f"Calibration — {res['name']}"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle("Reliability diagrams: raw vs Platt-scaled (calibrator fitted on a separate split)", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "calibration", "calibration_curves.png"), dpi=150)
plt.close()
log("Calibration curves saved.")

[HASEM] Step 6d: calibration curves (Platt scaling fitted on the calibration split)...
[HASEM] Calibration curves saved.


### 6e — Save outputs

In [42]:
sup_results = [res_lr, res_rf] + ([res_xgb] if res_xgb else [])
sup_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ["model", "proba", "pred", "cm"]} for r in sup_results])
sup_df.to_csv(os.path.join(OUTDIR, "supervised", "supervised_performance.csv"), index=False)
joblib.dump({r["name"]: r["model"] for r in sup_results}, os.path.join(OUTDIR, "models", "supervised_models.joblib"))
joblib.dump(platt_models, os.path.join(OUTDIR, "models", "platt_calibrators.joblib"))
log("Step 6 complete.")

[HASEM] Step 6 complete.


## Cell 7 — Multiclass cancer classification (patient-group split)

**7a.** Eligible primary tumours and labels.

In [43]:
log("Step 7: Multiclass classification with patient-group split...")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, classification_report
)

mask_tumour = (
    meta["partition"].eq("tumour").to_numpy()
    & meta["is_primary_tumour"].to_numpy()
)

tumour_idx = np.flatnonzero(mask_tumour)
y_tumour = (
    y_detailed.iloc[tumour_idx]
    .fillna("Unknown")
    .astype(str)
    .reset_index(drop=True)
)
group_tumour = (
    meta["patient_group"].iloc[tumour_idx]
    .astype(str)
    .to_numpy()
)

valid_group = (
    pd.notna(group_tumour)
    & (group_tumour != "")
    & (group_tumour != "nan")
)
tumour_idx = tumour_idx[valid_group]
y_tumour = y_tumour.iloc[np.flatnonzero(valid_group)].reset_index(drop=True)
group_tumour = group_tumour[valid_group]

group_table = pd.DataFrame({
    "group": group_tumour,
    "site": y_tumour.to_numpy()
}).groupby("group").agg(
    site=("site", lambda x: x.value_counts().index[0]),
    n_labels=("site", "nunique")
)

if (group_table["n_labels"] > 1).any():
    raise ValueError(
        "At least one patient group has multiple cancer labels. "
        "Resolve this before multiclass evaluation."
    )

group_counts = group_table["site"].value_counts()
valid_classes = group_counts[group_counts >= 5].index
keep_groups = group_table.index[
    group_table["site"].isin(valid_classes)
].to_numpy()

keep = np.isin(group_tumour, keep_groups)
tumour_idx = tumour_idx[keep]
y_tumour = y_tumour.iloc[np.flatnonzero(keep)].reset_index(drop=True)
group_tumour = group_tumour[keep]

[HASEM] Step 7: Multiclass classification with patient-group split...


### 7b — Patient-group split

In [44]:
# Split patient groups, not individual samples.
group_table = pd.DataFrame({
    "group": group_tumour,
    "site": y_tumour.to_numpy()
}).drop_duplicates("group").set_index("group")

group_ids = group_table.index.to_numpy()
group_sites = group_table["site"]

train_groups, test_groups = train_test_split(
    group_ids,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=group_sites.to_numpy()
)

train_mask = np.isin(group_tumour, train_groups)
test_mask = np.isin(group_tumour, test_groups)

X_mtr_idx = tumour_idx[train_mask]
X_mte_idx = tumour_idx[test_mask]
y_mtr_labels = y_tumour.to_numpy()[train_mask]
y_mte_labels = y_tumour.to_numpy()[test_mask]

le_multi = LabelEncoder()
y_mtr = le_multi.fit_transform(y_mtr_labels)
class_names = le_multi.classes_
class_to_int = {name: i for i, name in enumerate(class_names)}

# Do not silently drop test-only classes.
test_only = sorted(set(y_mte_labels) - set(class_names))
if test_only:
    raise ValueError(
        f"Test contains classes absent from training: {test_only}. "
        "Revise the patient-group split before reporting results."
    )

y_mte = np.array([class_to_int[x] for x in y_mte_labels])

train_patient_groups = set(group_tumour[train_mask])
test_patient_groups = set(group_tumour[test_mask])
overlap = train_patient_groups & test_patient_groups

assert not overlap, "Patient overlap remains in multiclass train/test split."

print("Multiclass patient-group overlap:", len(overlap))
print("Multiclass fit samples:", len(X_mtr_idx))
print("Multiclass test samples:", len(X_mte_idx))
print("Cancer classes:", len(class_names))

Multiclass patient-group overlap: 0
Multiclass fit samples: 6801
Multiclass test samples: 1701
Cancer classes: 28


### 7c — PCA on training tumours

In [45]:
# Fit PCA on training tumour samples only.
pca_multi = PCA(
    n_components=n_comp,
    svd_solver="full",
    random_state=RANDOM_SEED
)
pca_multi.fit(X_scaled.values[X_mtr_idx])

X_mtr_pca = pca_multi.transform(X_scaled.values[X_mtr_idx])
X_mte_pca = pca_multi.transform(X_scaled.values[X_mte_idx])

### 7d — Train and evaluate models

In [46]:
def train_multi(clf, Xtr, ytr, Xte, yte, name):
    clf.fit(Xtr, ytr)
    preds = clf.predict(Xte)

    acc = accuracy_score(yte, preds)
    f1 = f1_score(
        yte, preds, average="macro", zero_division=0
    )

    try:
        proba = clf.predict_proba(Xte)
        roc = roc_auc_score(
            yte,
            proba,
            labels=np.arange(len(class_names)),
            average="macro",
            multi_class="ovr"
        )
    except (ValueError, AttributeError):
        roc = np.nan

    print(
        f"  {name:22s}  ACC={acc:.4f}  "
        f"F1={f1:.4f}  ROC={roc:.4f}"
    )

    report = classification_report(
        yte,
        preds,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    return {
        "model": clf,
        "name": name,
        "acc": acc,
        "f1": f1,
        "roc": roc,
        "report": report
    }

n_classes = len(class_names)

print("\n[HASEM] Multiclass results:")

lr_m = train_multi(
    LogisticRegression(
        max_iter=5000,
        solver="saga",
        n_jobs=-1,
        random_state=RANDOM_SEED
    ),
    X_mtr_pca, y_mtr, X_mte_pca, y_mte,
    "Logistic Regression"
)

rf_m = train_multi(
    RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=RANDOM_SEED
    ),
    X_mtr_pca, y_mtr, X_mte_pca, y_mte,
    "Random Forest"
)

res_xgb_m = None
if HAVE_XGB:
    res_xgb_m = train_multi(
        xgb.XGBClassifier(
            n_estimators=300,
            objective="multi:softprob",
            num_class=n_classes,
            eval_metric="mlogloss",
            random_state=RANDOM_SEED,
            n_jobs=-1
        ),
        X_mtr_pca, y_mtr, X_mte_pca, y_mte,
        "XGBoost"
    )


[HASEM] Multiclass results:
  Logistic Regression     ACC=0.9718  F1=0.9441  ROC=0.9992
  Random Forest           ACC=0.9347  F1=0.8562  ROC=0.9969
  XGBoost                 ACC=0.9506  F1=0.9012  ROC=0.9960


### 7e — Save per-class reports

In [47]:
sup_multi_dir = os.path.join(OUTDIR, "supervised")
os.makedirs(sup_multi_dir, exist_ok=True)

for res in [lr_m, rf_m, res_xgb_m]:
    if res is None:
        continue

    tag = res["name"].replace(" ", "_")
    pd.DataFrame(res["report"]).T.to_csv(
        os.path.join(
            sup_multi_dir,
            f"{tag}_multiclass_report.csv"
        )
    )

log("Step 7 complete.")

[HASEM] Step 7 complete.


## Cell 8 — Gene contribution mapping and pathway enrichment

**8a.** Permutation importance on the binary random forest.

In [48]:
log("Step 8: Gene contribution mapping and pathway enrichment...")
contr_dir = os.path.join(OUTDIR, "contributions")
path_dir  = os.path.join(OUTDIR, "pathway_enrichment")

gene_names = list(X_scaled.columns)

log("8a: Computing permutation importance on binary Random Forest...")
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    res_rf["model"], X_te_pca, y_te,
    n_repeats=15,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
pc_importance = perm.importances_mean
log(f"Permutation importance computed for {len(pc_importance)} PCs.")

[HASEM] Step 8: Gene contribution mapping and pathway enrichment...
[HASEM] 8a: Computing permutation importance on binary Random Forest...
[HASEM] Permutation importance computed for 1330 PCs.


### 8b — Map PC importance to gene contributions

In [49]:
log("8b: Back-projecting PC importance to gene space...")
abs_loadings = np.abs(pca_sup.components_.T)   # (n_genes × n_PCs)
pc_imp       = np.array(pc_importance[:abs_loadings.shape[1]])
gene_scores  = abs_loadings.dot(pc_imp)

gene_contrib = pd.DataFrame({
    "gene_raw":      gene_names,
    "contribution":  gene_scores
}).sort_values("contribution", ascending=False).reset_index(drop=True)
gene_contrib["rank"] = range(1, len(gene_contrib) + 1)

[HASEM] 8b: Back-projecting PC importance to gene space...


### 8c — Strip Ensembl version numbers

In [50]:
def strip_version(eid):
    return str(eid).split(".")[0] if "." in str(eid) else str(eid)

gene_contrib["gene"] = gene_contrib["gene_raw"].apply(strip_version)
log(f"Example cleaned IDs: {gene_contrib['gene'].head(3).tolist()}")

[HASEM] Example cleaned IDs: ['ENSG00000109906', 'ENSG00000071991', 'ENSG00000236333']


### 8d — Ensembl to HGNC symbol conversion

In [51]:
log("8c: Converting Ensembl IDs to HGNC symbols...")
try:
    import mygene
    mg  = mygene.MyGeneInfo()
    ensg_ids = gene_contrib["gene"].tolist()

    # Query in batches of 1000 to avoid timeouts
    all_results = []
    for i in range(0, len(ensg_ids), 1000):
        batch = ensg_ids[i:i+1000]
        res   = mg.querymany(batch, scopes="ensembl.gene",
                             fields="symbol", species="human",
                             verbose=False)
        all_results.extend(res)

    conv_map = {}
    for r in all_results:
        if "symbol" in r:
            conv_map[r["query"]] = r["symbol"]
        else:
            conv_map[r["query"]] = r["query"]   # fallback to Ensembl ID

    gene_contrib["gene_hgnc"] = gene_contrib["gene"].map(conv_map).fillna(gene_contrib["gene"])
    n_converted = gene_contrib["gene_hgnc"].apply(
        lambda x: not x.startswith("ENSG")).sum()
    log(f"Converted {n_converted:,} of {len(gene_contrib):,} genes to HGNC symbols.")

except Exception as e:
    log(f"Gene conversion error: {e} — using Ensembl IDs directly.")
    gene_contrib["gene_hgnc"] = gene_contrib["gene"]

gene_contrib.to_csv(
    os.path.join(contr_dir, "gene_contributions_global.csv"), index=False)
log(f"Top 10 genes: {gene_contrib['gene_hgnc'].head(10).tolist()}")

[HASEM] 8c: Converting Ensembl IDs to HGNC symbols...
[HASEM] Converted 4,587 of 5,000 genes to HGNC symbols.
[HASEM] Top 10 genes: ['ZBTB16', 'CDH19', 'TRHDE-AS1', 'CA4', 'ADH1B', 'ADGRB3-DT', 'FAM153A', 'SFRP5', 'LCN10', 'PNMT']


### 8e — g:Profiler global enrichment (top 200 genes)

In [52]:
log("8d: Running g:Profiler global enrichment (top 200 genes)...")
from gprofiler import GProfiler
gp = GProfiler(return_dataframe=True)

top200 = gene_contrib.head(200)["gene_hgnc"].dropna().tolist()
top200_clean = [g for g in top200 if g and len(g) > 1]

log(f"Submitting {len(top200_clean)} genes to g:Profiler...")
try:
    enrich_global = gp.profile(
        organism  = "hsapiens",
        query     = top200_clean,
        sources   = ["GO:BP", "GO:MF", "KEGG", "REAC"],
        user_threshold = 0.05,
        significance_threshold_method = "fdr"
    )

    if enrich_global.empty:
        log("WARNING: g:Profiler returned no significant terms.")
    else:
        rename_map = {
            "source":            "Database",
            "name":              "Pathway/Term",
            "native":            "Term ID",
            "p_value":           "p-value",
            "intersection_size": "Overlap N",
            "term_size":         "Term Size",
            "precision":         "Precision",
            "recall":            "Recall"
        }
        enrich_global.rename(columns=rename_map, inplace=True, errors="ignore")

        if "p-value" in enrich_global.columns:
            pvals = enrich_global["p-value"].fillna(1.0).clip(0, 1).values
            from scipy.stats import false_discovery_control
            try:
                enrich_global["p-adj (FDR)"] = false_discovery_control(pvals)
            except Exception:
                # Manual Benjamini-Hochberg fallback
                n = len(pvals)
                order  = np.argsort(pvals)
                ranks  = np.empty(n); ranks[order] = np.arange(1, n+1)
                fdr    = np.minimum(1, pvals * n / ranks)
                enrich_global["p-adj (FDR)"] = fdr

        enrich_global = enrich_global.sort_values("p-value")
        enrich_global.to_csv(
            os.path.join(path_dir, "gprofiler_global_top200_FULL.csv"), index=False)

        log(f"Enrichment complete: {len(enrich_global)} significant terms found.")
        print("\n[HASEM] Top 15 enriched pathways:")
        disp_cols = ["Database","Pathway/Term","Overlap N",
                     "p-value","p-adj (FDR)","Precision"]
        disp_cols = [c for c in disp_cols if c in enrich_global.columns]
        print(enrich_global.head(15)[disp_cols].to_string(index=False))

except Exception as e:
    log(f"g:Profiler enrichment error: {e}")
    enrich_global = pd.DataFrame()

[HASEM] 8d: Running g:Profiler global enrichment (top 200 genes)...
[HASEM] Submitting 200 genes to g:Profiler...
[HASEM] Enrichment complete: 241 significant terms found.

[HASEM] Top 15 enriched pathways:
Database                       Pathway/Term  Overlap N      p-value  p-adj (FDR)  Precision
   GO:BP              muscle system process         21 3.384041e-09 8.155538e-07   0.148936
   GO:BP   multicellular organismal process         88 3.916418e-08 4.719283e-06   0.624113
   GO:BP                 muscle contraction         17 1.095314e-07 5.547739e-06   0.120567
   GO:BP   regulation of membrane potential         19 1.095314e-07 5.547739e-06   0.134752
    REAC                 Muscle contraction         15 1.150983e-07 5.547739e-06   0.153061
    REAC        Striated Muscle Contraction          8 1.561760e-07 6.273068e-06   0.081633
   GO:BP                 system development         59 2.715358e-07 9.348589e-06   0.418440
    KEGG       Cytoskeleton in muscle cells         16 3.

### 8f — Per-cancer enrichment (top 3 cancer types by size)

In [53]:
log("8e: Running per-cancer enrichment for largest tumour types...")
top_cancers = (y_detailed[y_binary==1]
               .value_counts().head(3).index.tolist())

for cancer in top_cancers:
    try:
        mask_c  = (y_detailed == cancer).values
        Xc      = X_scaled.values[mask_c]
        Xc_pca  = pca_sup.transform(Xc)

        # Use RF feature importances as a proxy for this cancer type
        perm_c  = permutation_importance(
            res_rf["model"], Xc_pca[:min(200, len(Xc_pca))],
            y_binary.values[mask_c][:min(200, len(Xc_pca))],
            n_repeats=5, random_state=RANDOM_SEED, n_jobs=-1)

        gc = abs_loadings.dot(perm_c.importances_mean[:abs_loadings.shape[1]])
        gdf = pd.DataFrame({"gene":gene_names,"contribution":gc})
        gdf = gdf.sort_values("contribution", ascending=False)

        top_g = gdf.head(200)["gene"].apply(strip_version).tolist()
        top_g_hgnc = [conv_map.get(g, g) for g in top_g]
        top_g_clean = [g for g in top_g_hgnc if g and len(g) > 1]

        r = gp.profile(
            organism  = "hsapiens",
            query     = top_g_clean,
            sources   = ["GO:BP", "KEGG", "REAC"],
            user_threshold = 0.05,
            significance_threshold_method = "fdr"
        )
        tag = cancer.replace("/","_").replace(" ","_")
        r.to_csv(os.path.join(path_dir,
                 f"gprofiler_{tag}_top200.csv"), index=False)
        log(f"  {cancer}: {len(r)} enriched terms saved.")
    except Exception as ex:
        log(f"  {cancer}: enrichment failed ({ex})")

log("Step 8 complete.")

[HASEM] 8e: Running per-cancer enrichment for largest tumour types...
[HASEM]   Breast: 82 enriched terms saved.
[HASEM]   Lung: 130 enriched terms saved.
[HASEM]   Kidney: 302 enriched terms saved.
[HASEM] Step 8 complete.


## Cell 9 — Final summary

All numbers for the manuscript (v2, leakage-safe).

In [54]:
print("=" * 70)
print("  HASEM v2 (LEAKAGE-SAFE) — COMPLETE RESULTS SUMMARY")
print("=" * 70)
print("\n── DATASET AND PARTITIONS ──────────────────────────────────")
print(f"  Total samples        : {len(y_binary):,}")
print(f"  Healthy              : {int((y_binary==0).sum()):,}  "
      f"(train {int(healthy_train_mask.sum()):,} / calibration {int(healthy_calib_mask.sum()):,} / holdout {int(healthy_holdout_mask.sum()):,})")
print(f"  Tumour               : {int((y_binary==1).sum()):,}  (evaluation tumours: {int(eval_tumour_mask.sum()):,})")
print(f"  Evaluation pool      : {int(eval_mask.sum()):,}")
print(f"  Genes scanned (train): {len(mad_df):,} | retained: {len(top_genes):,}")
print(f"  PCA components       : {n_comp}")
print(f"  Control-limit pct    : {THRESH_PCT} | ComBat mode: {COMBAT_MODE} | eval primary only: {EVAL_PRIMARY_ONLY}")
print(f"  Limits               : T2={T2_thresh_emp:.3f}  Q={Q_thresh_emp:.3f}")

print("\n── PROVENANCE ──────────────────────────────────────────────")
print(f"  GTEx vs TCGA/TARGET-normal separability (healthy only): ROC-AUC {batch_auc:.4f}")
print(fa_src.to_string(index=False))

print("\n── UNSUPERVISED DETECTION (evaluation pool) ────────────────")
for r in [res_emp, res_theo, res_iso]:
    print(f"  {r['label']:30s}  ROC={r['roc']:.4f}  PR={r['pr']:.4f}  Det={r['detect']:.2f}%  FA={r['false_alarm']:.2f}%")
print("\n── ABLATION (evaluation pool) ──────────────────────────────")
print(abl_df[["label", "roc", "pr", "detect", "false_alarm"]].round(4).to_string(index=False))

print("\n── PER-CANCER DETECTION (lowest 8, Q alone) ────────────────")
print(pc_Q.tail(8)[["Cancer Type", "N Samples", "Detection Rate (%)", "95% CI Lower", "95% CI Upper"]].to_string(index=False))
print("\n── LOCO-C (matching tissue removed; lowest 8) ──────────────")
if len(df_true_loco):
    print(df_true_loco.tail(8).to_string(index=False))

print("\n── SUPERVISED CLASSIFICATION (untouched test set) ──────────")
for res in [res_lr, res_rf, res_xgb]:
    if res is None: continue
    print(f"  {res['name']:22s}  ROC={res['roc']:.4f}  PR={res['pr']:.4f}  Acc={res['acc']:.4f}  "
          f"F1={res['f1']:.4f}  Brier={res['brier']:.4f}  Brier(Platt)={res.get('brier_platt', float('nan')):.4f}")

print("\n── PATHWAY ENRICHMENT (top 10 global) ──────────────────────")
if "enrich_global" in globals() and enrich_global is not None and not enrich_global.empty:
    cols = [c for c in ["Database", "Pathway/Term", "Overlap N", "p-value", "p-adj (FDR)", "Precision"] if c in enrich_global.columns]
    print(enrich_global.head(10)[cols].to_string(index=False))
else:
    print("  (see pathway_enrichment/ directory)")
print("\n  Everything is saved in:", OUTDIR)
print("  Deposit with the paper: models/split_manifest.csv, supervised_split_manifest.csv,")
print("  config_snapshot.csv, this notebook, and all CSV/PNG outputs.")
print("=" * 70)

print("=== HASEM v3 FINAL MODEL VERIFICATION ===")

print("Genes retained:", len(top_genes))
print("PCA components retained:", pca_unsup.n_components_)
print(
    "Cumulative variance explained:",
    round(pca_unsup.explained_variance_ratio_.sum() * 100, 4), "%"
)
print("Configured variance target:", VAR_EXPLAINED * 100, "%")

print("\n=== EMPIRICAL CONTROL LIMITS ===")
print("Threshold percentile:", THRESH_PCT)
print("T² threshold:", T2_thresh_emp)
print("Q threshold:", Q_thresh_emp)

print("\n=== EVALUATION COUNTS ===")
print("Healthy training:", int(healthy_train_mask.sum()))
print("Healthy calibration:", int(healthy_calib_mask.sum()))
print("Healthy holdout:", int(healthy_holdout_mask.sum()))
print("Tumour evaluation:", int(eval_tumour_mask.sum()))

print("\n=== PATIENT OVERLAP CHECK ===")
print("Train overlap:", len(eval_patients & healthy_train_patients))
print("Calibration overlap:", len(eval_patients & healthy_calib_patients))
print("Holdout overlap:", len(eval_patients & healthy_holdout_patients))

  HASEM v2 (LEAKAGE-SAFE) — COMPLETE RESULTS SUMMARY

── DATASET AND PARTITIONS ──────────────────────────────────
  Total samples        : 17,330
  Healthy              : 8,149  (train 5,688 / calibration 1,188 / holdout 1,273)
  Tumour               : 9,181  (evaluation tumours: 8,502)
  Evaluation pool      : 9,775
  Genes scanned (train): 25,937 | retained: 5,000
  PCA components       : 1330
  Control-limit pct    : 99.0 | ComBat mode: none | eval primary only: False
  Limits               : T2=1645.288  Q=1329.645

── PROVENANCE ──────────────────────────────────────────────
  GTEx vs TCGA/TARGET-normal separability (healthy only): ROC-AUC nan
                 Group    N  FA_OR_pct  FA_OR_CI_lo  FA_OR_CI_hi  FA_Qonly_pct  FA_Q_CI_lo  FA_Q_CI_hi
GTEx healthy (holdout) 1161       2.07         1.39         3.06          1.72        1.12        2.65
TCGA healthy (holdout)  112       6.25         3.06        12.34          0.89        0.16        4.88

── UNSUPERVISED DETECTION (evalu